# Probability Distribution & Naive Bayes

### Using Probability Distributions for Data Simulation and Classification

## Problem Statement

Different types of birds can have different characteristics such as weight, wingspan, and beak length. Probability distributions can be used to simulate these characteristics before applying a classification method.

In this case, Naive Bayes is used to predict the bird type based on the simulated characteristics.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

## Probability Distributions

The simulated bird characteristics are generated using different probability distributions.

For continuous characteristics such as wingspan and weight, a Gaussian distribution is used.

In [2]:
np.random.seed(42)

wingspan = np.random.normal(45, 5, 1000)
weight = np.random.normal(500, 100, 1000)

print(wingspan[:5])
print(weight[:5])

[47.48357077 44.30867849 48.23844269 52.61514928 43.82923313]
[639.93554366 592.46336829 505.96303699 435.30632223 569.82233136]


For discrete characteristics such as the number of days a bird sings in a month, a binomial distribution is used.

In [3]:
sing_days = np.random.binomial(30, 0.6, 1000)

print(sing_days[:5])

[19 22 19 21 16]


For the beak-to-head ratio, a uniform distribution is used to generate values within a defined range.

In [4]:
beak_head_ratio = np.random.uniform(0.2, 0.5, 1000)

print(beak_head_ratio[:5])

[0.29072585 0.36902251 0.44114147 0.24114455 0.37420973]


## Simulated Bird Data

The simulated features are combined into a dataset representing different bird characteristics.

In [5]:
df_birds = pd.DataFrame({
    'wingspan_cm': wingspan,
    'weight_g': weight,
    'sing_days': sing_days,
    'beak_head_ratio': beak_head_ratio
})

df_birds.head()

,wingspan_cm,weight_g,sing_days,beak_head_ratio
0,47.483571,639.935544,19,0.290726
1,44.308678,592.463368,22,0.369023
2,48.238443,505.963037,19,0.441141
3,52.615149,435.306322,21,0.241145
4,43.829233,569.822331,16,0.374210


## Bird Type

The dataset is divided into three simulated bird types, represented by the `breed` label.

In [7]:
FEATURES = ['wingspan_cm', 'weight_g', 'sing_days', 'beak_head_ratio']

In [8]:
breed_params = {
    0: {
        'wingspan_cm': (35, 1.5),
        'weight_g': (20, 1),
        'sing_days': (30, 0.8),
        'beak_head_ratio': (0.1, 0.6)
    },
    1: {
        'wingspan_cm': (30, 2),
        'weight_g': (25, 5),
        'sing_days': (30, 0.5),
        'beak_head_ratio': (0.2, 0.5)
    },
    2: {
        'wingspan_cm': (40, 3.5),
        'weight_g': (32, 3),
        'sing_days': (30, 0.3),
        'beak_head_ratio': (0.1, 0.3)
    }
}

## Generate Data for Each Bird Type

Each bird type is generated using its own probability distribution parameters. This allows the simulated features to have different patterns across the three bird types.

In [9]:
def generate_breed_data(breed, n_samples):
    params = breed_params[breed]

    data = pd.DataFrame({
        'wingspan_cm': np.random.normal(
            params['wingspan_cm'][0],
            params['wingspan_cm'][1],
            n_samples
        ),
        'weight_g': np.random.normal(
            params['weight_g'][0],
            params['weight_g'][1],
            n_samples
        ),
        'sing_days': np.random.binomial(
            params['sing_days'][0],
            params['sing_days'][1],
            n_samples
        ),
        'beak_head_ratio': np.random.uniform(
            params['beak_head_ratio'][0],
            params['beak_head_ratio'][1],
            n_samples
        )
    })

    data['breed'] = breed

    return data

In [10]:
df_0 = generate_breed_data(0, 1200)
df_1 = generate_breed_data(1, 1350)
df_2 = generate_breed_data(2, 900)

df_birds = pd.concat([df_0, df_1, df_2])
df_birds = df_birds.sample(frac=1, random_state=42).reset_index(drop=True)

df_birds.head()

,wingspan_cm,weight_g,sing_days,beak_head_ratio,breed
0,31.719262,25.207709,13,0.404669,1
1,35.719741,20.114101,24,0.297745,0
2,36.658750,20.380778,28,0.272393,0
3,27.837173,24.610386,11,0.247384,1
4,34.385298,18.170318,23,0.209941,0


## Train-Test Split

The dataset is split into training and testing sets to evaluate the Naive Bayes classifier on data that was not used during training.

In [11]:
split = int(len(df_birds) * 0.7)

df_train = df_birds[:split].reset_index(drop=True)
df_test = df_birds[split:].reset_index(drop=True)

print('Training data:', len(df_train))
print('Testing data:', len(df_test))

Training data: 2415
Testing data: 1035


## Naive Bayes Classification

The Naive Bayes classifier is used to predict the bird type based on its characteristics.

In [12]:
X_train = df_train[FEATURES]
y_train = df_train['breed']

X_test = df_test[FEATURES]
y_test = df_test['breed']

In [13]:
y_train.value_counts()

breed
1    955
0    835
2    625
Name: count, dtype: int64

## Training the Naive Bayes Model

The training data is used to estimate the probability distribution of each feature for every bird type.

In [14]:
train_params = {}

for breed in sorted(y_train.unique()):
    data = df_train[df_train['breed'] == breed]
    
    train_params[breed] = {
        'wingspan_cm': (
            data['wingspan_cm'].mean(),
            data['wingspan_cm'].std()
        ),
        'weight_g': (
            data['weight_g'].mean(),
            data['weight_g'].std()
        ),
        'sing_days': (
            30,
            data['sing_days'].mean() / 30
        ),
        'beak_head_ratio': (
            data['beak_head_ratio'].min(),
            data['beak_head_ratio'].max()
        )
    }

train_params

{np.int64(0): {'wingspan_cm': (np.float64(34.98899638627496),
   np.float64(1.494961355286117)),
  'weight_g': (np.float64(19.936506718865214), np.float64(1.0083208218274482)),
  'sing_days': (30, np.float64(0.8015169660678644)),
  'beak_head_ratio': (np.float64(0.10012043725893324),
   np.float64(0.5987568728309479))},
 np.int64(1): {'wingspan_cm': (np.float64(30.01478587870019),
   np.float64(1.9696748082869375)),
  'weight_g': (np.float64(24.715780513526976), np.float64(4.76918271808795)),
  'sing_days': (30, np.float64(0.49787085514834206)),
  'beak_head_ratio': (np.float64(0.20007573634716863),
   np.float64(0.499977448049953))},
 np.int64(2): {'wingspan_cm': (np.float64(39.99575386842279),
   np.float64(3.5470786000389554)),
  'weight_g': (np.float64(32.02483300012886), np.float64(2.962000493131283)),
  'sing_days': (30, np.float64(0.3058666666666667)),
  'beak_head_ratio': (np.float64(0.10050435234056401),
   np.float64(0.2994689098566996))}}

## Class Probabilities

The probability of each bird type is estimated from its proportion in the training data.

In [15]:
class_probs = y_train.value_counts(normalize=True).sort_index()

class_probs

breed
0    0.345756
1    0.395445
2    0.258799
Name: proportion, dtype: float64

## Feature Probabilities

For each bird type, the probability of each feature is calculated based on the distribution estimated from the training data.

In [16]:
def calculate_feature_probability(row, breed):
    params = train_params[breed]

    wingspan_prob = stats.norm.pdf(
        row['wingspan_cm'],
        params['wingspan_cm'][0],
        params['wingspan_cm'][1]
    )

    weight_prob = stats.norm.pdf(
        row['weight_g'],
        params['weight_g'][0],
        params['weight_g'][1]
    )

    sing_prob = stats.binom.pmf(
        row['sing_days'],
        params['sing_days'][0],
        params['sing_days'][1]
    )

    beak_prob = stats.uniform.pdf(
        row['beak_head_ratio'],
        params['beak_head_ratio'][0],
        params['beak_head_ratio'][1] - params['beak_head_ratio'][0]
    )

    return wingspan_prob, weight_prob, sing_prob, beak_prob

## Naive Bayes Prediction

The posterior probability for each bird type is calculated by multiplying the class probability with the probability of all observed features.

The bird is classified into the breed with the highest posterior probability.

In [17]:
def predict_breed(row):
    probabilities = {}

    for breed in class_probs.index:
        feature_probs = calculate_feature_probability(row, breed)

        posterior = class_probs[breed]

        for probability in feature_probs:
            posterior *= probability

        probabilities[breed] = posterior

    return max(probabilities, key=probabilities.get)

In [18]:
y_pred = df_test.apply(predict_breed, axis=1)

y_pred.head()

0    2
1    2
2    1
3    2
4    1
dtype: int64

## Model Evaluation

The model performance is evaluated by comparing the predicted breed with the actual breed in the testing data.

In [19]:
accuracy = (y_pred.values == y_test.values).mean()

print('Test Accuracy:', accuracy)

Test Accuracy: 0.996135265700483


In [20]:
confusion_matrix = pd.crosstab(
    y_test,
    y_pred,
    rownames=['Actual'],
    colnames=['Predicted']
)

confusion_matrix

Predicted,0,1,2
Actual,,,
0,364,1,0
1,2,393,0
2,0,1,274


## Conclusion

The analysis demonstrates how probability distributions and Naive Bayes can be used for classification.

The model estimated the probability distribution of each feature for each bird type and used these probabilities to classify unseen observations. The resulting test accuracy of 99.61% shows that the approach works well on the simulated dataset.